In [1]:
import duckdb
import pandas as pd

con = duckdb.connect("faultlines.duckdb", read_only=True)

# 1. Check which (Date, Ticker) pairs appear more than once
query_dupes = """
    SELECT Ticker, Date, COUNT(*) as record_count
    FROM raw_prices
    GROUP BY Ticker, Date
    HAVING COUNT(*) > 1
    ORDER BY record_count DESC
    LIMIT 10
"""
dupe_summary = con.execute(query_dupes).df()
print("--- Sample Duplicate Ticker Dates ---")
display(dupe_summary)

--- Sample Duplicate Ticker Dates ---


,Ticker,Date,record_count


In [2]:
if not dupe_summary.empty:
  sample_ticker = dupe_summary.iloc[0]["Ticker"]
  sample_date = dupe_summary.iloc[0]["Date"]

  query_details = f"""
        SELECT filename, Date, Ticker, Open, High, Low, Close, Volume
        FROM raw_prices
        WHERE Ticker = '{sample_ticker}' AND Date = '{sample_date}'
    """
  display(con.execute(query_details).df())

In [3]:
con.close()

In [4]:
import duckdb

con = duckdb.connect("faultlines.duckdb", read_only=True)

query = """
    SELECT 
        Ticker, 
        Asset_Type, 
        COUNT(Close) as trading_days
    FROM raw_prices
    WHERE Date BETWEEN '2016-01-01' AND '2016-12-31'
      AND Asset_Type = 'Stock'
    GROUP BY Ticker, Asset_Type
    ORDER BY trading_days DESC
    LIMIT 10
"""

display(con.execute(query).df())
con.close()

,Ticker,Asset_Type,trading_days
0,ens.us,Stock,252
1,cldt.us,Stock,252
2,dlng_a.us,Stock,252
3,cys.us,Stock,252
4,advm.us,Stock,252
5,ead.us,Stock,252
6,arc.us,Stock,252
7,fult.us,Stock,252
8,gsat.us,Stock,252
9,plce.us,Stock,252


In [5]:
from data_pipeline import get_log_return_matrix, get_ticker_metadata

returns_df = get_log_return_matrix(
    "2016-01-01", "2016-12-31", asset_type="Stock"
)
meta_df = get_ticker_metadata(tickers=list(returns_df.columns))

print(f"Matrix Shape: {returns_df.shape}")
print(f"Metadata Mapped: {len(meta_df)} tickers")
display(meta_df.head(10))

Matrix Shape: (270, 5092)
Metadata Mapped: 5092 tickers


,Ticker,Asset_Type,Company_Name,cik_str
1,bpmx,Stock,NaN,<NA>
3,iart,Stock,INTEGRA LIFESCIENCES HOLDINGS CORP,917520
5,nyld,Stock,NaN,<NA>
7,pkbk,Stock,"PARKE BANCORP, INC.",1315399
8,unfi,Stock,UNITED NATURAL FOODS INC,1020859
10,zen,Stock,NaN,<NA>
13,acls,Stock,AXCELIS TECHNOLOGIES INC,1113232
15,cslt,Stock,NaN,<NA>
16,ctrn,Stock,Citi Trends Inc,1318484
17,cvv,Stock,CVD EQUIPMENT CORP,766792


In [7]:
import numpy as np
import pandas as pd
from data_pipeline import get_log_return_matrix, get_ticker_metadata
from data_quality_checks import run_data_quality_checks

# 1. Run the full quality suite
run_data_quality_checks()

         FAULTLINES DATA PIPELINE - DATA QUALITY SUITE      

1. Primary Key Integrity (Date, Ticker):
   -> Duplicate (Date, Ticker) pairs found: 0 [PASSED]

2. Price Sanity Check (Close/Open <= 0):
   -> Invalid price rows: 0 [PASSED]

3. High/Low Price Consistency Check (High < Low):
   -> Inconsistent High/Low rows: 0 [PASSED]

4. SEC Metadata Linkage Coverage:
   -> Mapped Tickers: 2,633 / 5,092 (51.71%)

5. Extreme Single-Day Log Return Outliers:
   -> Returns > +50%: 222 instances
   -> Returns < -50%: 74 instances

6. Matrix Sparsity Check:
   -> Filled Zero Entries: 129,868 / 1,374,840 (9.45%)

                  DATA QUALITY AUDIT COMPLETE               
